<a href="https://colab.research.google.com/github/bhaktii1802/vision-deployment-pipeline/blob/main/Pytorch_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Thu Nov 26 04:49:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install onnx onnxruntime-gpu opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 81.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 MB 74.1 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnx]


In [9]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 30.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxscript]


In [10]:
import torch, torchvision

device = "cuda"
model = torchvision.models.resnet18(weights="DEFAULT").eval().to(device)
dummy = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    model, dummy, "resnet18.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=17,
)
print("exported")

/tmp/ipykernel_5082/666560746.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1126 04:59:38.487000 5082 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:55: 

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported


In [11]:
import onnxruntime as ort, numpy as np

In [13]:
import onnxruntime as ort, numpy as np

sess = ort.InferenceSession("resnet18.onnx", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
print(sess.get_providers())  # check whether CUDA is listed first

onnx_out = sess.run(None, {"input": dummy.cpu().numpy()})[0]
torch_out = model(dummy).detach().cpu().numpy()
print("max diff:", np.abs(onnx_out - torch_out).max())

['CUDAExecutionProvider', 'CPUExecutionProvider']
max diff: 4.053116e-06


In [14]:
import time

x_np = dummy.cpu().numpy()

# PyTorch (GPU)
for _ in range(20): model(dummy)          # warm-up
torch.cuda.synchronize()
t = time.time()
for _ in range(100): model(dummy)
torch.cuda.synchronize()
print("PyTorch ms:", (time.time() - t) * 10)

# ONNX Runtime
for _ in range(20): sess.run(None, {"input": x_np})
t = time.time()
for _ in range(100): sess.run(None, {"input": x_np})
print("ONNX ms:", (time.time() - t) * 10)

PyTorch ms: 7.272956371307373
ONNX ms: 3.7175369262695312


In [15]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/vision-deployment
!cp resnet18.onnx /content/drive/MyDrive/vision-deployment/
!ls -lh /content/drive/MyDrive/vision-deployment/

Mounted at /content/drive
total 88K
-rw------- 1 root root 88K Nov 26 05:13 resnet18.onnx


In [16]:


def bench_torch(n=100):
    for _ in range(20): model(dummy)
    torch.cuda.synchronize()
    t = time.time()
    for _ in range(n): model(dummy)
    torch.cuda.synchronize()
    return (time.time() - t) / n * 1000

def bench_onnx(n=100):
    for _ in range(20): sess.run(None, {"input": x_np})
    t = time.time()
    for _ in range(n): sess.run(None, {"input": x_np})
    return (time.time() - t) / n * 1000

torch_runs = [bench_torch() for _ in range(5)]
onnx_runs  = [bench_onnx()  for _ in range(5)]

print(f"PyTorch: {np.mean(torch_runs):.2f} ± {np.std(torch_runs):.2f} ms")
print(f"ONNX RT: {np.mean(onnx_runs):.2f} ± {np.std(onnx_runs):.2f} ms")
print(f"Speedup: {np.mean(torch_runs)/np.mean(onnx_runs):.2f}x")

PyTorch: 4.73 ± 0.73 ms
ONNX RT: 2.59 ± 0.04 ms
Speedup: 1.83x
